# Mean Variance Optimization with CVXOPT and CVXPY

In [5]:
import sys
sys.path.insert(0,'/Users/martincaroe/Desktop/MARTIN/Copenhagen Business School/Msc Economics and Business Administration - Finance and Investments /3 Semester/Python for the Financial Economics/Fork/python_for_the_financial_economist')


# import relevant packages
import numpy as np
import pandas as pd
from scipy import stats, optimize
import matplotlib.pyplot as plt

# packages for convex optimization
import cvxpy as cp
import cvxopt

# own functions from codelib
from codelib.statistics import moments as mom
from codelib.portfolio_optimization.mean_variance import portfolio_mean, portfolio_std, portfolio_variance, minimum_variance_portfolio

from codelib.visualization.layout import DefaultStyle
DefaultStyle();

Consider the portfolio optimization problem 


where the expected return vector is given by 

$$
\boldsymbol{\mu} = \begin{pmatrix} 0.032 \\ 0.0322 \\ 0.084 \\ 0.082 \end{pmatrix},
$$

the vector of volatilites is given by 

$$
\mathbf{v} = \begin{pmatrix} 0.05 \\ 0.05 \\ 0.22 \\ 0.22 \end{pmatrix},
$$

and the correlation matrix is given by 

$$
\mathbf{C} = \begin{pmatrix} 1.0 & 0.85 & 0.5 & 0.45 \\
                      0.85 & 1.0 & 0.5 & 0.45 \\
                      0.5 & 0.5 & 1.0 & 0.9 \\
                      0.45 & 0.45 & 0.9 & 1.0 \end{pmatrix}
$$

In [6]:
corr_mat = np.array([[1.0, 0.85, 0.5, 0.45],
                     [0.85, 1.0, 0.5, 0.45],
                     [0.5, 0.5, 1.0, 0.9],
                     [0.45, 0.45, 0.9, 1.0]])

vols = np.array([5.0, 5.0, 22.0, 22.0]) / 100.0
mu = np.array([3.2, 3.22, 8.4, 8.2]) / 100.0

cov_mat = mom.corr_to_cov_matrix(corr_mat, vols)

## Problem 1

Set up the minimization problem that enable you to find the minimum variance portfolio using both `cvxopt` and `cvxpy`. Apply a budget constraint requiring the portfolio weights to sum to one. Compare with the analytical solution. 

In [9]:
"""
CVXPY
"""
w = cp.Variable(4)                                        # 4 weights to solve for

prob = cp.Problem(cp.Minimize(cp.quad_form(w, cov_mat)),  # minimize w' * Sigma * w
                  [cp.sum(w) == 1])                       # budget constraint
prob.solve()

w_cvxpy = w.value

"""
CVXOPT
"""
P = cvxopt.matrix(2 * cov_mat)
q = cvxopt.matrix(np.zeros(4))
A = cvxopt.matrix(np.ones((1, 4)))       # 1' w = 1
b = cvxopt.matrix([1.0])

sol = cvxopt.solvers.qp(P, q, A=A, b=b)
w_cvxopt = np.array(sol['x']).flatten()

"""
Analytical solution (formula from Lecture 5)
"""
ones = np.ones(4)
cov_inv = np.linalg.inv(cov_mat)
w_analytical = cov_inv @ ones / (ones @ cov_inv @ ones)

**Tolkning:** All three methods (the CVXPY solver, the CVXOPT solver, and the analytical formula) give the same minimum-variance portfolio, so we can trust the setup. Notice that CVXPY needed no tuning at all, unlike the general-purpose SciPy optimizer, because this problem is convex and CVXPY is built for exactly that kind of problem.

## Problem 2

Minimize the portfolio variance subject to the constraints that using the `CVXPY` or `CVXOPT` package. 

* The expected return should be above 6%
* No shorting (all weights should be equal or greater than 0)
* The portfolio weights need to sum to one (budget constraint)

In [8]:
"""
CVXPY
"""
w = cp.Variable(4)

prob = cp.Problem(cp.Minimize(cp.quad_form(w, cov_mat)),
                  [cp.sum(w) == 1,          # budget
                   w >= 0,                  # no shorting
                   mu @ w >= 0.06])         # expected return at least 6%
prob.solve()

w_opt = w.value
print(w_opt)
print('Expected return:', mu @ w_opt)
print('Std:', np.sqrt(prob.value))          # prob.value is the minimized variance

[0.10444791 0.35063358 0.34197387 0.20294464]
Expected return: 0.06000000000000002
Std: 0.12967804947852904


**Tolkning:** All the solvers agree with the SciPy answer from the other exercise: to earn at least 6% without shorting, the minimum risk we can achieve is about 13%, with roughly 55% of the money in the two risky assets. The difference between the tools is only in how convenient they are: CVXPY lets us write the requirements almost like plain English (`w >= 0`, `mu @ w >= 0.06`), while CVXOPT and SciPy need us to rewrite them into a fixed format first.